# 4장. 피벗테이블과 VLOOKUP을 코드로

정리한 표로 보고서에 들어갈 숫자를 만든다. 지점별 합계, 목표 대비 달성률, 전주 대비, 지점×품목군 표.

In [1]:
from pathlib import Path

import pandas as pd

DATA = Path("../data")
WORK = Path("../work")

sales = pd.read_csv(WORK / "03_clean.csv", parse_dates=["판매일"])
prev_sales = pd.read_csv(WORK / "03_clean_prev.csv", parse_dates=["판매일"])
items = pd.read_excel(DATA / "master" / "품목.xlsx")
targets = pd.read_excel(DATA / "master" / "지점목표.xlsx")

print(len(sales), "건 /", len(prev_sales), "건")
targets

253 건 / 252 건


,지점,주간목표
0,강남점,7500000
1,수원점,6500000
2,대전점,5500000
3,부산점,7500000
4,광주점,4500000


## 1. 지점별 합계
엑셀 피벗테이블에서 행에 '지점', 값에 '금액 합계'를 넣은 것과 같다.

In [2]:
by_branch = sales.groupby("지점", as_index=False)["금액"].sum()
by_branch

,지점,금액
0,강남점,8004800
1,광주점,4858200
2,대전점,4060700
3,부산점,7020900
4,수원점,5975100


## 2. 목표 붙이기 (VLOOKUP)
`merge`가 VLOOKUP이다. 목표 표를 왼쪽에 두면 지점 순서가 목표 파일에 적힌 순서로 고정된다.
`validate`는 지점 이름이 한쪽에 두 번 나오면 멈추게 하는 안전장치다.

In [3]:
summary = targets.merge(by_branch, on="지점", how="left", validate="one_to_one")
summary["달성률"] = summary["금액"] / summary["주간목표"]
summary

,지점,주간목표,금액,달성률
0,강남점,7500000,8004800,1.067307
1,수원점,6500000,5975100,0.919246
2,대전점,5500000,4060700,0.738309
3,부산점,7500000,7020900,0.936120
4,광주점,4500000,4858200,1.079600


## 3. 전주 대비

In [4]:
prev = (
    prev_sales.groupby("지점", as_index=False)["금액"].sum()
    .rename(columns={"금액": "전주금액"})
)
summary = summary.merge(prev, on="지점", how="left", validate="one_to_one")
summary["전주대비"] = summary["금액"] / summary["전주금액"] - 1
summary = summary[["지점", "금액", "주간목표", "달성률", "전주금액", "전주대비"]]
summary

,지점,금액,주간목표,달성률,전주금액,전주대비
0,강남점,8004800,7500000,1.067307,7118800,0.124459
1,수원점,5975100,6500000,0.919246,5894600,0.013657
2,대전점,4060700,5500000,0.738309,6494400,-0.374738
3,부산점,7020900,7500000,0.936120,8488400,-0.172883
4,광주점,4858200,4500000,1.079600,3793800,0.280563


소수로 두면 읽기 어렵다. 화면에서만 퍼센트로 바꿔 본다. 표 안의 값은 그대로 소수다.

In [5]:
summary.style.format({
    "금액": "{:,}", "주간목표": "{:,}", "전주금액": "{:,}",
    "달성률": "{:.1%}", "전주대비": "{:+.1%}",
}).hide(axis="index")

지점,금액,주간목표,달성률,전주금액,전주대비
강남점,"8,004,800","7,500,000",106.7%,"7,118,800",+12.4%
수원점,"5,975,100","6,500,000",91.9%,"5,894,600",+1.4%
대전점,"4,060,700","5,500,000",73.8%,"6,494,400",-37.5%
부산점,"7,020,900","7,500,000",93.6%,"8,488,400",-17.3%
광주점,"4,858,200","4,500,000",108.0%,"3,793,800",+28.1%


## 4. 품목군 붙이기
판매 내역에는 품목군이 없다. 품목 표에서 품목코드로 찾아 붙인다.
붙인 뒤에는 못 찾은 줄이 있는지 반드시 센다.

In [6]:
with_group = sales.merge(
    items[["품목코드", "품목군"]], on="품목코드", how="left", validate="many_to_one"
)
print("품목군을 못 찾은 줄:", with_group["품목군"].isna().sum())
print("붙이기 전후 줄 수:", len(sales), "→", len(with_group))
with_group.head(3)

품목군을 못 찾은 줄: 0
붙이기 전후 줄 수: 253 → 253


,지점,원본파일,전표번호,판매일,품목코드,품목명,수량,단가,금액,품목군
0,강남점,강남점_주간판매.xlsx,GN-1102-001,2026-11-02,P-303,컵라면 24개,5,21600,108000,식음료
1,강남점,강남점_주간판매.xlsx,GN-1102-002,2026-11-02,P-403,USB 충전 케이블,2,5900,11800,전자소모품
2,강남점,강남점_주간판매.xlsx,GN-1102-003,2026-11-02,P-202,종이컵 1000개,4,15400,61600,생활용품


## 5. 지점×품목군 표

In [7]:
pivot = with_group.pivot_table(
    index="지점", columns="품목군", values="금액",
    aggfunc="sum", fill_value=0, margins=True, margins_name="합계",
)
pivot = pivot.reindex(list(targets["지점"]) + ["합계"]).fillna(0).astype("int64")
pivot

품목군,사무용품,생활용품,식음료,전자소모품,합계
지점,,,,,
강남점,1321000,1708400,2730900,2244500,8004800
수원점,2126700,1097900,1147100,1603400,5975100
대전점,1254100,801800,951700,1053100,4060700
부산점,3154800,1854100,1091200,920800,7020900
광주점,1577400,1291500,837800,1151500,4858200
합계,9434000,6753700,6758700,6973300,29919700


## 6. 많이 팔린 품목 5개

In [8]:
top_items = (
    sales.groupby(["품목코드", "품목명"], as_index=False)[["수량", "금액"]].sum()
    .sort_values("금액", ascending=False)
    .head(5)
)
top_items

,품목코드,품목명,수량,금액
0,P-101,A4 복사용지 1박스,268,6566000
10,P-402,멀티탭 4구,327,4087500
8,P-303,컵라면 24개,148,3196800
4,P-202,종이컵 1000개,189,2910600
6,P-301,믹스커피 100개,198,2732400


## 검산
세 표의 전체 합계가 모두 판매 내역 합계와 같아야 한다.

In [9]:
total = sales["금액"].sum()
print(f"판매 내역 합계  {total:,}")
print(f"지점별 요약 합계 {summary['금액'].sum():,}", summary["금액"].sum() == total)
print(f"품목군 표 합계  {pivot.loc['합계', '합계']:,}", pivot.loc["합계", "합계"] == total)

판매 내역 합계  29,919,700
지점별 요약 합계 29,919,700 True
품목군 표 합계  29,919,700 True


In [10]:
summary.to_csv(WORK / "04_summary.csv", index=False, encoding="utf-8-sig")
pivot.to_csv(WORK / "04_pivot.csv", encoding="utf-8-sig")
top_items.to_csv(WORK / "04_top_items.csv", index=False, encoding="utf-8-sig")

## 실습. 요일별 판매 금액
엑셀이라면 요일 열을 하나 만들고 피벗을 다시 돌려야 한다. 여기서는 두 줄이다.

In [11]:
days = ["월", "화", "수", "목", "금", "토", "일"]
by_day = sales.groupby(sales["판매일"].dt.day_of_week)["금액"].sum()
by_day.index = [days[i] for i in by_day.index]
by_day

월    4013500
화    5466300
수    4136300
목    5240100
금    7022800
토    4040700
Name: 금액, dtype: int64